<a href="https://colab.research.google.com/github/ortiqboltayev94-dotcom/NUUz-3D-Cyberpunk-Map/blob/main/aral_sea_timelapse_1984_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q earthengine-api geemap imageio[ffmpeg] matplotlib pandas pillow

In [2]:
GEE_PROJECT_ID = "project-4affa162-d806-49f2-9c5"  # Undo bosgandan keyingi aniq ID

In [11]:
import os
import io
import time
import json
import requests
import ee
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import imageio.v2 as imageio
from PIL import Image

# ==========================================
# 1. SOZLAMALAR
# ==========================================
GEE_PROJECT_ID = "project-4affa162-d806-49f2-9c5"
OUTPUT_DIR = "./aral_30day_map"
FRAMES_DIR = os.path.join(OUTPUT_DIR, "frames_cache")
STATS_JSON = os.path.join(OUTPUT_DIR, "stats_cache.json")
FINAL_MP4 = os.path.join(OUTPUT_DIR, "aral_sea_1984_2026_30DayMapChallenge.mp4")
CSV_STATS = os.path.join(OUTPUT_DIR, "aral_area_stats_1984_2026.csv")

START_YEAR = 1984
END_YEAR = 2026
FPS = 4
ARAL_BBOX = [58.0, 43.2, 62.0, 46.9]


def init_gee(project_id: str) -> None:
    ee.Initialize(project=project_id)
    print("[OK] GEE ulandi.")


def prep_landsat(img: ee.Image, is_l89: bool = False) -> ee.Image:
    qa = img.select("QA_PIXEL")
    mask = qa.bitwiseAnd(1 << 3).eq(0).And(qa.bitwiseAnd(1 << 4).eq(0))
    bands = ["SR_B3", "SR_B4", "SR_B5", "SR_B6"] if is_l89 else ["SR_B2", "SR_B3", "SR_B4", "SR_B5"]
    return (
        img.select(bands, ["GREEN", "RED", "NIR", "SWIR1"])
        .multiply(0.0000275)
        .add(-0.2)
        .updateMask(mask)
    )


def get_annual_composite(year: int, aoi: ee.Geometry):
    start = ee.Date.fromYMD(year, 5, 1)
    end = ee.Date.fromYMD(year, 9, 30)

    if year <= 2012:
        l5 = ee.ImageCollection("LANDSAT/LT05/C02/T1_L2").filterBounds(aoi).filterDate(start, end)
        l7 = ee.ImageCollection("LANDSAT/LE07/C02/T1_L2").filterBounds(aoi).filterDate(start, end)
        col = l5.merge(l7).map(lambda i: prep_landsat(i, False))
    else:
        # 2013-2026: Landsat 8 + Landsat 9 (tez, barqaror va yagona 30m standart)
        l8 = ee.ImageCollection("LANDSAT/LC08/C02/T1_L2").filterBounds(aoi).filterDate(start, end)
        l9 = ee.ImageCollection("LANDSAT/LC09/C02/T1_L2").filterBounds(aoi).filterDate(start, end)
        col = l8.merge(l9).map(lambda i: prep_landsat(i, True))

    if col.size().getInfo() == 0:
        return None
    return col.median().clip(aoi)


def compute_year_data_with_retry(year: int, aoi: ee.Geometry, retries: int = 4):
    for attempt in range(1, retries + 1):
        try:
            comp = get_annual_composite(year, aoi)
            if comp is None:
                return None, None, None

            mndwi = comp.normalizedDifference(["GREEN", "SWIR1"]).rename("MNDWI")
            ndvi = comp.normalizedDifference(["NIR", "RED"]).rename("NDVI")

            water_mask = mndwi.gt(0.05)
            veg_mask = ndvi.gt(0.15).And(water_mask.Not())

            pixel_area = ee.Image.pixelArea().divide(1e6)
            water_area_img = pixel_area.updateMask(water_mask).rename("water_km2")
            veg_area_img = pixel_area.updateMask(veg_mask).rename("veg_km2")

            stats = water_area_img.addBands(veg_area_img).reduceRegion(
                reducer=ee.Reducer.sum(),
                geometry=aoi,
                scale=500,
                maxPixels=1e10,
                tileScale=4,
                bestEffort=True,
            ).getInfo()

            water_km2 = float(stats.get("water_km2") or 0.0)
            veg_km2 = float(stats.get("veg_km2") or 0.0)

            bg = ee.Image.constant([22, 25, 30]).rename(["vis-red", "vis-green", "vis-blue"]).toUint8().clip(aoi)
            water_vis = mndwi.updateMask(water_mask).visualize(min=0.05, max=0.6, palette=["#023e8a", "#0077b6", "#00b4d8", "#90e0ef"])
            veg_vis = ndvi.updateMask(veg_mask).visualize(min=0.15, max=0.55, palette=["#1b4332", "#2d6a4f", "#52b788", "#95d5b2"])

            mosaic = ee.ImageCollection([bg, veg_vis, water_vis]).mosaic()

            url = mosaic.getThumbURL({
                "region": aoi,
                "dimensions": 720,
                "format": "png",
                "crs": "EPSG:3857",
            })
            resp = requests.get(url, timeout=90)
            resp.raise_for_status()
            map_img = Image.open(io.BytesIO(resp.content)).convert("RGB")
            return map_img, water_km2, veg_km2

        except Exception as e:
            wait_sec = attempt * 6
            print(f"  [QAYTA URINISH {attempt}/{retries}] {year}-yil: {e}. {wait_sec} soniya kutilmoqda...")
            time.sleep(wait_sec)

    print(f"  [OGOHLANTIRISH] {year}-yil {retries} urinishda ham yuklanmadi, o'tkazib yuborildi.")
    return None, None, None


def render_combined_frame(year: int, map_img: Image.Image, df: pd.DataFrame, max_w: float, max_v: float) -> np.ndarray:
    fig = plt.figure(figsize=(16, 9), dpi=120, facecolor="#16191e")
    gs = fig.add_gridspec(2, 2, width_ratios=[1.25, 1], height_ratios=[1, 1], wspace=0.15, hspace=0.35)

    ax_map = fig.add_subplot(gs[:, 0])
    ax_map.imshow(map_img)
    ax_map.axis("off")
    ax_map.set_title(f"OROL DENGIZI VA OROLBO'YI DINAMIKASI — {year}", color="white", fontsize=18, fontweight="bold", pad=12, loc="left")

    cur = df[df["year"] <= year]
    cur_row = df[df["year"] == year].iloc[0]

    ax_w = fig.add_subplot(gs[0, 1], facecolor="#1e2229")
    ax_w.plot(cur["year"], cur["water_km2"], color="#00b4d8", linewidth=2.5)
    ax_w.fill_between(cur["year"], cur["water_km2"], color="#00b4d8", alpha=0.2)
    ax_w.scatter([year], [cur_row["water_km2"]], color="#90e0ef", s=60, zorder=5)
    ax_w.set_xlim(START_YEAR, END_YEAR)
    ax_w.set_ylim(0, max_w * 1.1)
    ax_w.set_title(f"Suv yuzasi (MNDWI): {cur_row['water_km2']:,.0f} km²", color="#90e0ef", fontsize=13, fontweight="bold", loc="left")
    ax_w.tick_params(colors="#cccccc")
    ax_w.grid(color="#2e3440", linestyle="--", linewidth=0.6)

    ax_v = fig.add_subplot(gs[1, 1], facecolor="#1e2229")
    ax_v.plot(cur["year"], cur["veg_km2"], color="#52b788", linewidth=2.5)
    ax_v.fill_between(cur["year"], cur["veg_km2"], color="#52b788", alpha=0.2)
    ax_v.scatter([year], [cur_row["veg_km2"]], color="#95d5b2", s=60, zorder=5)
    ax_v.set_xlim(START_YEAR, END_YEAR)
    ax_v.set_ylim(0, max_v * 1.15)
    ax_v.set_title(f"O'simlik va saksovulzor (NDVI): {cur_row['veg_km2']:,.0f} km²", color="#95d5b2", fontsize=13, fontweight="bold", loc="left")
    ax_v.tick_params(colors="#cccccc")
    ax_v.grid(color="#2e3440", linestyle="--", linewidth=0.6)

    fig.text(0.02, 0.02, "Manba: USGS Landsat 5/7/8/9 (GEE)  |  #30DayMapChallenge", color="#9aa0a6", fontsize=11)

    buf = io.BytesIO()
    plt.savefig(buf, format="png", facecolor=fig.get_facecolor())
    plt.close(fig)
    buf.seek(0)
    return imageio.imread(buf)


def main() -> None:
    os.makedirs(FRAMES_DIR, exist_ok=True)
    init_gee(GEE_PROJECT_ID)
    aoi = ee.Geometry.Rectangle(ARAL_BBOX)

    if os.path.exists(STATS_JSON):
        with open(STATS_JSON, "r", encoding="utf-8") as f:
            stats_cache = json.load(f)
    else:
        stats_cache = {}

    years = list(range(START_YEAR, END_YEAR + 1))

    for y in years:
        y_str = str(y)
        frame_path = os.path.join(FRAMES_DIR, f"{y}.png")

        if os.path.exists(frame_path) and y_str in stats_cache:
            print(f"[CACHE] {y}-yil tayyor, o'tkazib yuborildi.")
            continue

        print(f"[INFO] {y}-yil qayta ishlanmoqda...")
        img, w_km2, v_km2 = compute_year_data_with_retry(y, aoi)
        if img is None:
            continue

        img.save(frame_path)
        stats_cache[y_str] = {"year": y, "water_km2": w_km2, "veg_km2": v_km2}
        with open(STATS_JSON, "w", encoding="utf-8") as f:
            json.dump(stats_cache, f, indent=2)

    records = sorted(stats_cache.values(), key=lambda x: x["year"])
    df = pd.DataFrame(records)
    df.to_csv(CSV_STATS, index=False)
    print(f"[OK] Statistika saqlandi: {CSV_STATS}")

    max_w = max(df["water_km2"].max(), 1.0)
    max_v = max(df["veg_km2"].max(), 1.0)

    print("[INFO] MP4 video yig'ilmoqda...")
    with imageio.get_writer(FINAL_MP4, fps=FPS, codec="libx264", macro_block_size=None) as writer:
        for y in df["year"]:
            y_int = int(y)
            map_img = Image.open(os.path.join(FRAMES_DIR, f"{y_int}.png")).convert("RGB")
            frame = render_combined_frame(y_int, map_img, df, max_w, max_v)
            writer.append_data(frame)

    print(f"[TAYYOR] Video saqlandi: {FINAL_MP4}")


if __name__ == "__main__":
    main()

[OK] GEE ulandi.
[CACHE] 1984-yil tayyor, o'tkazib yuborildi.
[CACHE] 1985-yil tayyor, o'tkazib yuborildi.
[CACHE] 1986-yil tayyor, o'tkazib yuborildi.
[CACHE] 1987-yil tayyor, o'tkazib yuborildi.
[CACHE] 1988-yil tayyor, o'tkazib yuborildi.
[CACHE] 1989-yil tayyor, o'tkazib yuborildi.
[CACHE] 1990-yil tayyor, o'tkazib yuborildi.
[CACHE] 1991-yil tayyor, o'tkazib yuborildi.
[CACHE] 1992-yil tayyor, o'tkazib yuborildi.
[CACHE] 1993-yil tayyor, o'tkazib yuborildi.
[CACHE] 1994-yil tayyor, o'tkazib yuborildi.
[CACHE] 1995-yil tayyor, o'tkazib yuborildi.
[CACHE] 1996-yil tayyor, o'tkazib yuborildi.
[CACHE] 1997-yil tayyor, o'tkazib yuborildi.
[CACHE] 1998-yil tayyor, o'tkazib yuborildi.
[CACHE] 1999-yil tayyor, o'tkazib yuborildi.
[CACHE] 2000-yil tayyor, o'tkazib yuborildi.
[CACHE] 2001-yil tayyor, o'tkazib yuborildi.
[CACHE] 2002-yil tayyor, o'tkazib yuborildi.
[CACHE] 2003-yil tayyor, o'tkazib yuborildi.
[CACHE] 2004-yil tayyor, o'tkazib yuborildi.
[CACHE] 2005-yil tayyor, o'tkazib yubo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [6]:
import ee
ee.Authenticate(auth_mode="notebook")

To authorize access needed by Earth Engine, open the following URL in a web browser and follow the instructions. If the web browser does not start automatically, please manually browse the URL below.

    https://code.earthengine.google.com/client-auth?scopes=https%3A//www.googleapis.com/auth/earthengine%20https%3A//www.googleapis.com/auth/cloud-platform%20https%3A//www.googleapis.com/auth/drive%20https%3A//www.googleapis.com/auth/devstorage.full_control&request_id=x0-lA1IuhsOY0MjuL6DMBUt6ZuEMbRF7Jhsof7IgV-Q&tc=_xgeCxZtWoRvdrDbfymr6sIee6uKqBRfTyc57eBeAtY&cc=WQZmWjmCNR_b-cLQW-AGvNi8H75M4V2ujnpA1AzKIDo

The authorization workflow will generate a code, which you should paste in the box below.
Enter verification code: 4/1AUn6Hd_HG76OXicS-ORvtBagvtbVxv5cNmz9_TXHA5ECGQ_d6SLRm66cc8Y

Successfully saved authorization token.


In [ ]:
4/1AUn6Hd_RRynrvLbT8LRlJ2e9onem79_cUK-7H6DaD9YQvxM5u96DpidHzQI